# Gabarito — Módulo 5: Bidirectional LSTM

In [1]:
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

X_train_pad = np.load("X_train_pad.npy")
X_test_pad = np.load("X_test_pad.npy")
y_train = np.load("y_train.npy")
y_test = np.load("y_test.npy")
embedding_matrix = np.load("embedding_matrix.npy")
with open("config.json") as f:
    config = json.load(f)
test_text = pd.read_csv("test_text.csv")

print(X_train_pad.shape, X_test_pad.shape, config)
predictions_rnn = pd.read_csv("predictions_rnn.csv")
predictions_lstm = pd.read_csv("predictions_lstm.csv")

(75, 15) (25, 15) {'maxlen': 15, 'vocab_size': 225, 'embed_dim': 32}


In [2]:
# 5.1
X_train_t = torch.tensor(X_train_pad, dtype=torch.long)
X_test_t = torch.tensor(X_test_pad, dtype=torch.long)
y_train_t = torch.tensor(y_train, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.float32)

In [3]:
# 5.2
embed_dim = config["embed_dim"]

class SMSBiLSTM(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = nn.Embedding.from_pretrained(
            torch.tensor(embedding_matrix, dtype=torch.float32),
            freeze=False,
            padding_idx=0,
        )
        self.lstm = nn.LSTM(
            input_size=embed_dim, hidden_size=32, batch_first=True, bidirectional=True
        )
        self.fc = nn.Linear(32 * 2, 1)

    def forward(self, x):
        emb = self.embedding(x)
        out, (hidden, cell) = self.lstm(emb)
        combined = torch.cat([hidden[0], hidden[1]], dim=1)
        logits = self.fc(combined)
        return torch.sigmoid(logits).squeeze(1)

torch.manual_seed(1)
model = SMSBiLSTM()

In [4]:
# 5.3
loss_fn = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

for epoch in range(60):
    optimizer.zero_grad()
    y_proba_train = model(X_train_t)
    loss = loss_fn(y_proba_train, y_train_t)
    loss.backward()
    optimizer.step()
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch + 1}, loss={loss.item():.4f}")

Epoch 10, loss=0.1448
Epoch 20, loss=0.0070
Epoch 30, loss=0.0011
Epoch 40, loss=0.0005
Epoch 50, loss=0.0003
Epoch 60, loss=0.0002


In [5]:
# 5.4
model.eval()
with torch.no_grad():
    y_proba = model(X_test_t).numpy()
y_pred = (y_proba >= 0.5).astype(int)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, pos_label=1, zero_division=0)
recall = recall_score(y_test, y_pred, pos_label=1, zero_division=0)
f1 = f1_score(y_test, y_pred, pos_label=1, zero_division=0)

print(f"Accuracy: {accuracy:.2f}")
print(f"Precision: {precision:.2f}")
print(f"Recall: {recall:.2f}")
print(f"F1-score: {f1:.2f}")

Accuracy: 1.00
Precision: 1.00
Recall: 1.00
F1-score: 1.00


In [6]:
# 5.5
results = pd.DataFrame({
    "text": test_text["clean_text"],
    "y_true": y_test,
    "y_pred": y_pred,
    "y_proba": y_proba,
})
results.to_csv("predictions_bilstm.csv", index=False)
results.head()

,text,y_true,y_pred,y_proba
0,you have been selected to win a free voucher c...,1,1,0.999715
1,coffee tomorrow morning before work,0,0,0.000281
2,claim your free cash prize now urgent reply ne...,1,1,0.999642
3,win free cash now click link urgent claim requ...,1,1,0.999708
4,free cash prize waiting call now to claim urge...,1,1,0.999681


In [7]:
# 5.6
comparison = pd.DataFrame({
    "model": ["RNN", "LSTM", "Bidirectional LSTM"],
    "f1_score": [
        f1_score(predictions_rnn["y_true"], predictions_rnn["y_pred"], pos_label=1, zero_division=0),
        f1_score(predictions_lstm["y_true"], predictions_lstm["y_pred"], pos_label=1, zero_division=0),
        f1_score(y_test, y_pred, pos_label=1, zero_division=0),
    ],
})
comparison

,model,f1_score
0,RNN,0.7
1,LSTM,1.0
2,Bidirectional LSTM,1.0


## 5.7 Reflexão final (exemplo de resposta)

Nessa base, a diferença apareceu de forma clara: a RNN simples ficou bem
atrás (F1 ≈ 0.70, loss de treino instável, "pulando" em vez de cair de
forma suave), enquanto tanto a LSTM quanto a Bidirectional LSTM acertaram
100% do conjunto de teste. As mensagens mais difíceis — como SMS de "ham"
que usam palavras tipicamente associadas a spam ("free", "win", "call
urgent") em contextos inofensivos — são exatamente onde a RNN simples
mais erra: seu hidden state tende a "esquecer" o contexto da frase inteira
e reagir mais à presença isolada de certas palavras, enquanto os
mecanismos de portão (gates) da LSTM ajudam o modelo a reter melhor esse
contexto mais longo.

Com uma base bem maior, a expectativa é que essa vantagem da LSTM/BiLSTM
sobre a RNN simples se mantenha ou aumente — mas a diferença entre a LSTM
"comum" e a Bidirectional LSTM tende a ficar mais evidente também, já que
ler a frase nos dois sentidos ajuda mais quando há mais variedade de
construções de frase para aprender. Com poucos exemplos como aqui, LSTM e
BiLSTM já dão conta do recado sozinhas, então a bidirecionalidade não teve
chance de mostrar toda a sua vantagem.